# Lecture 8: Generative Models — GAN, VAE, and Diffusion


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
print("Libraries loaded.")


## 1. The GAN Idea (Ch. 15)

Two networks play against each other:
- **Generator (G)**: Generate realistic data from noise → fool the discriminator
- **Discriminator (D)**: Distinguish real from fake

$$\\min_G \\max_D \\mathbb{E}_{x\\sim p_{data}}[\\log D(x)] + \\mathbb{E}_{z\\sim p_z}[\\log(1-D(G(z)))]$$


In [ ]:
# GAN training: learn a 1D Gaussian distribution
class SimpleGAN1D:
    def __init__(self, latent_dim=1, target_mean=3.0, target_std=0.8):
        self.target_mean = target_mean
        self.target_std  = target_std
        # Generator: z -> x (2 layers)
        self.Wg1 = np.random.randn(8, latent_dim) * 0.3
        self.bg1 = np.zeros(8)
        self.Wg2 = np.random.randn(1, 8) * 0.3
        self.bg2 = np.zeros(1)
        # Discriminator: x -> prob (2 layers)
        self.Wd1 = np.random.randn(8, 1) * 0.3
        self.bd1 = np.zeros(8)
        self.Wd2 = np.random.randn(1, 8) * 0.3
        self.bd2 = np.zeros(1)
    
    def _relu(self, x): return np.maximum(0, x)
    def _sigmoid(self, x): return 1/(1+np.exp(-np.clip(x,-15,15)))
    
    def generate(self, z):
        h = self._relu(z @ self.Wg1.T + self.bg1)
        return h @ self.Wg2.T + self.bg2
    
    def discriminate(self, x):
        h = self._relu(x @ self.Wd1.T + self.bd1)
        return self._sigmoid(h @ self.Wd2.T + self.bd2)
    
    def sample_real(self, n):
        return np.random.normal(self.target_mean, self.target_std, (n, 1))
    
    def sample_latent(self, n):
        return np.random.randn(n, 1)

gan = SimpleGAN1D()
lr = 0.005
batch_size = 64
n_epochs = 300
d_losses, g_losses = [], []

for epoch in range(n_epochs):
    for _ in range(2):  # 2x Discriminator update
        z = gan.sample_latent(batch_size)
        x_fake = gan.generate(z)
        x_real = gan.sample_real(batch_size)
        d_real = gan.discriminate(x_real)
        d_fake = gan.discriminate(x_fake)
        d_loss = -np.mean(np.log(d_real + 1e-8)) - np.mean(np.log(1 - d_fake + 1e-8))
        # Simple hand-coded update (gradient approximation)
        gan.Wd2 -= lr * np.random.randn(*gan.Wd2.shape) * d_loss * 0.1
    
    z = gan.sample_latent(batch_size)
    x_fake = gan.generate(z)
    d_fake = gan.discriminate(x_fake)
    g_loss = -np.mean(np.log(d_fake + 1e-8))
    gan.Wg2 -= lr * np.random.randn(*gan.Wg2.shape) * g_loss * 0.1
    gan.Wg1 -= lr * 0.5 * np.random.randn(*gan.Wg1.shape) * g_loss * 0.05
    
    d_losses.append(d_loss); g_losses.append(g_loss)

z_test = gan.sample_latent(1000)
x_gen = gan.generate(z_test).flatten()
x_real_test = gan.sample_real(1000).flatten()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(d_losses, 'steelblue', lw=1.5, label="Discriminator loss", alpha=0.7)
axes[0].plot(g_losses, 'crimson',   lw=1.5, label="Generator loss", alpha=0.7)
axes[0].set_title("GAN Training Losses"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(x_real_test, bins=40, alpha=0.6, color='steelblue', density=True, label="Real data")
axes[1].hist(x_gen,       bins=40, alpha=0.6, color='crimson',   density=True, label="Generated data")
x_range = np.linspace(x_real_test.min(), x_real_test.max(), 200)
axes[1].plot(x_range, norm.pdf(x_range, 3.0, 0.8), 'k--', lw=2, label="Target distribution")
axes[1].set_title("Generated vs Real Data Distribution"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("1D GAN Example", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb08_gan.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Mode Collapse and GAN Challenges (Ch. 15.2)

GAN training can be unstable:
- **Mode collapse**: The generator learns only one mode (no diversity)
- **Vanishing gradients**: If the discriminator is too good, the generator's gradient approaches zero

**Solutions**: Wasserstein GAN, gradient penalty, feature matching


In [ ]:
# Mode collapse simulation - multimodal distribution
np.random.seed(5)
def sample_real_multimodal(n):
    modes = [(-3, 0.5), (0, 0.3), (3, 0.5)]
    choices = np.random.choice(len(modes), n)
    samples = np.array([np.random.normal(modes[c][0], modes[c][1]) for c in choices])
    return samples

x_real_mm = sample_real_multimodal(2000)
x_mode_collapse = np.random.normal(-3, 0.4, 2000)  # mode collapse: only mode 0
x_good_gen = sample_real_multimodal(2000) + np.random.randn(2000) * 0.1  # all 3 modes

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
bins = np.linspace(-6, 6, 50)

axes[0].hist(x_real_mm, bins=bins, density=True, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title("Real Data: 3-Modal Distribution"); axes[0].grid(alpha=0.3)

axes[1].hist(x_mode_collapse, bins=bins, density=True, color='crimson', alpha=0.8, edgecolor='white')
axes[1].set_title("Mode Collapse: GAN learned only 1 mode"); axes[1].grid(alpha=0.3)

axes[2].hist(x_good_gen, bins=bins, density=True, color='limegreen', alpha=0.8, edgecolor='white')
axes[2].set_title("Good GAN: learned all 3 modes"); axes[2].grid(alpha=0.3)

plt.suptitle("Mode Collapse Problem", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb08_mode_collapse.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. VAE: Latent Variable Model (Ch. 17)

A VAE consists of two networks:
- **Encoder** `q(z|x)`: Encode data into latent space → produce μ and σ
- **Decoder** `p(x|z)`: Reconstruct data from the latent vector

**ELBO (Evidence Lower Bound):**
$$\\mathcal{L} = \\underbrace{\\mathbb{E}_{q(z|x)}[\\log p(x|z)]}_{\\text{Reconstruction}} - \\underbrace{D_{KL}(q(z|x) \\| p(z))}_{\\text{Regularizer}}$$


In [ ]:
# VAE two terms: reconstruction loss + KL divergence
def kl_divergence_normal(mu, log_var):
    return -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))

def reconstruction_loss_mse(x, x_hat):
    return np.mean((x - x_hat)**2)

mu_range = np.linspace(-3, 3, 100)
log_var_range = np.linspace(-4, 2, 100)
MU, LV = np.meshgrid(mu_range, log_var_range)
KL = -0.5 * (1 + LV - MU**2 - np.exp(LV))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im = axes[0].contourf(MU, LV, KL, levels=30, cmap='hot')
axes[0].set_xlabel("mu"); axes[0].set_ylabel("log_var")
axes[0].set_title("KL Divergence: q(z|x) || N(0,1)")
plt.colorbar(im, ax=axes[0])
axes[0].contour(MU, LV, KL, levels=[0], colors='white', linewidths=2)
axes[0].text(0, 0, "KL=0\n(N(0,1))", ha='center', fontsize=10, color='white', fontweight='bold')

beta_values = [0.0, 0.5, 1.0, 2.0]
recon_terms = [0.8, 0.6, 0.45, 0.3]
kl_terms = [0.0, 0.1, 0.25, 0.5]

x_pos = np.arange(len(beta_values))
width = 0.35
axes[1].bar(x_pos - width/2, recon_terms, width, label="Reconstruction loss", color='steelblue')
axes[1].bar(x_pos + width/2, kl_terms,    width, label="KL Divergence", color='crimson')
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f"beta={b}" for b in beta_values])
axes[1].set_title("beta-VAE: Loss Term Balance")
axes[1].set_ylabel("Loss value"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb08_vae_loss.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Reparameterization Trick (Ch. 17.7)

Stochastic sampling for backpropagation must be made deterministic:

$$z = \\mu + \\sigma \\odot \\epsilon, \\quad \\epsilon \\sim \\mathcal{N}(0, I)$$

This allows gradients to flow through μ and σ.


In [ ]:
def vae_encode(x, W_enc_mu, W_enc_lv, b_enc_mu, b_enc_lv):
    mu      = np.tanh(x @ W_enc_mu + b_enc_mu)
    log_var = x @ W_enc_lv + b_enc_lv
    return mu, log_var

def reparameterize(mu, log_var):
    eps = np.random.randn(*mu.shape)
    return mu + np.exp(0.5 * log_var) * eps

def vae_decode(z, W_dec, b_dec):
    return np.tanh(z @ W_dec + b_dec)

np.random.seed(11)
n_classes = 5
n_per_class = 50
colors_cls = ['steelblue', 'crimson', 'limegreen', 'darkorange', 'purple']
d_in = 8; d_latent = 2

W_enc_mu = np.random.randn(d_in, d_latent) * 0.5
W_enc_lv = np.random.randn(d_in, d_latent) * 0.3
b_enc_mu = np.zeros(d_latent)
b_enc_lv = np.ones(d_latent) * -2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cls in range(n_classes):
    x_cls = np.random.randn(n_per_class, d_in) + cls * 0.5
    mu_cls, lv_cls = vae_encode(x_cls, W_enc_mu, W_enc_lv, b_enc_mu, b_enc_lv)
    z_cls = reparameterize(mu_cls, lv_cls)
    
    axes[0].scatter(mu_cls[:, 0], mu_cls[:, 1], alpha=0.7, s=20, color=colors_cls[cls], label=f"Class {cls}")
    axes[1].scatter(z_cls[:, 0],  z_cls[:, 1],  alpha=0.7, s=20, color=colors_cls[cls])

for ax, title in zip(axes, ["Latent Means (mu)", "Sampled z via Reparameterization"]):
    ax.set_title(title); ax.set_xlabel("z1"); ax.set_ylabel("z2")
    ax.grid(alpha=0.3)

axes[0].legend(fontsize=8)
plt.suptitle("VAE Latent Space: Mean vs Sampled Points", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb08_vae_latent.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Diffusion Models (Ch. 18)

### Forward Process
Data is progressively corrupted with Gaussian noise:
$$q(x_t | x_{t-1}) = \\mathcal{N}(x_t; \\sqrt{1-\\beta_t} x_{t-1}, \\beta_t I)$$

### Reverse Process
A trained network predicts the noise at each step to recover the original data.


In [ ]:
# Diffusion forward process - 1D example
T = 50
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

np.random.seed(3)
x0_data = np.concatenate([
    np.random.normal(-2, 0.4, 300),
    np.random.normal( 2, 0.4, 300),
])

def forward_diffusion(x0, t):
    alpha_bar_t = alpha_bars[t]
    noise = np.random.randn(*x0.shape)
    xt = np.sqrt(alpha_bar_t) * x0 + np.sqrt(1 - alpha_bar_t) * noise
    return xt, noise

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
time_steps = [0, 5, 10, 20, 35, T-1]

for ax, t in zip(axes, time_steps):
    xt, _ = forward_diffusion(x0_data, t)
    ax.hist(xt, bins=40, density=True, color='steelblue', alpha=0.8, edgecolor='white')
    x_range = np.linspace(-4, 4, 100)
    ax.plot(x_range, norm.pdf(x_range, 0, 1), 'r--', lw=1.5, label="N(0,1)")
    ab = alpha_bars[t]
    ax.set_title(f"t={t}\nSNR={ab/(1-ab):.2f}")
    ax.set_xlim(-5, 5); ax.set_ylim(0, 1.5)
    if t == 0: ax.set_xlabel("Real data")
    if t == T-1: ax.set_xlabel("Pure noise")

plt.suptitle("Diffusion Forward Process: From Data to Noise", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb08_diffusion_fwd.png", dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Noise schedule visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
t_range = np.arange(T)

axes[0].plot(t_range, betas, 'steelblue', lw=2)
axes[0].set_title("Noise Schedule (betas)"); axes[0].set_xlabel("t"); axes[0].grid(alpha=0.3)

axes[1].plot(t_range, alpha_bars, 'crimson', lw=2)
axes[1].set_title("Cumulative Alpha (alpha_bar)"); axes[1].set_xlabel("t"); axes[1].grid(alpha=0.3)
axes[1].set_ylabel("Remaining signal power")

axes[2].plot(t_range, np.sqrt(alpha_bars), 'steelblue', lw=2, label="sqrt(alpha_bar) ~ signal")
axes[2].plot(t_range, np.sqrt(1-alpha_bars), 'crimson', lw=2, label="sqrt(1-alpha_bar) ~ noise")
axes[2].set_title("Signal vs Noise Ratio"); axes[2].set_xlabel("t")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb08_noise_schedule.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Generative Model Comparison

| Model | Training | Sampling | Advantage | Disadvantage |
|---|---|---|---|---|
| **GAN** | Adversarial | Fast (single pass) | Sharp images | Unstable training, mode collapse |
| **VAE** | ELBO max. | Fast (single pass) | Structured latent space | Blurry outputs |
| **Diffusion** | Noise prediction | Slow (T steps) | High quality, diversity | Slow sampling |

## 7. Summary

| Concept | Description |
|---|---|
| **GAN** | Learning via generator + discriminator competition |
| **Mode collapse** | GAN fails to learn all modes |
| **VAE ELBO** | Reconstruction + KL balance |
| **Reparam. trick** | z=μ+σε enables gradient flow |
| **Diffusion forward** | Progressively corrupt data with noise |
| **Diffusion reverse** | Progressively remove noise |

**Next Notebook →** Reinforcement Learning
